<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Layout algebra

A **layout** is a `(Shape, Stride)` pair, and it is a *function*: it maps a logical
coordinate to a linear index. That one idea is the spine of FlyDSL — every tiled
copy and every MMA in the earlier notebooks was really "lay out the data, partition
it across threads, then move or multiply." Before we tile and swizzle (next
notebook), this one builds layouts, maps coordinates, cuts layouts into tiles, and
meets the two kinds of tensor.

Everything here is **compile-time layout algebra**. We still wrap each cell in a
`@flyc.jit` because constructing a layout needs an MLIR context (the same reason
arithmetic lived inside a kernel in `01_numeric_types`), but nothing launches on the
GPU — the layout objects just print as readable values.

In [ ]:
import os

# These cells print at *trace* time. A warm JIT disk cache would skip the re-trace, and
# the prints would vanish on a re-run — so disable the disk cache (the kernels are tiny).
# Set before importing flydsl.
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"

import flydsl.compiler as flyc
import flydsl.expr as fx

## 1. Shape, Stride, Layout

The **shape** is the logical extent of each mode; the **stride** is how far you step
in linear memory per unit of that mode. A row-major 8×8 is shape `(8, 8)` with
stride `(8, 1)` — moving one row jumps 8 elements, moving one column jumps 1.

A layout reports a handful of properties: `size` (number of coordinates), `cosize`
(how many distinct indices it produces), `rank` (number of modes), and `depth` (how
deeply the shape is nested).

In [ ]:
@flyc.jit
def layout_basics():
    L = fx.make_layout((8, 8), (8, 1))
    print("layout      :", L)
    print("shape/stride:", fx.get_shape(L), "/", fx.get_stride(L))
    print("size/cosize :", fx.size(L), "/", fx.cosize(L))
    print("rank/depth  :", fx.rank(L), "/", fx.depth(L))


layout_basics()

## 2. Coordinates → indices

`crd2idx` applies the layout: it takes a coordinate and returns the linear index.
The shape decides which coordinates are legal; the **stride** decides where each one
lands. So the *same* coordinate gives a *different* index under a row-major vs a
column-major layout — that difference is the whole point of carrying a stride.

In [ ]:
@flyc.jit
def coord_to_index():
    row = fx.make_layout((8, 8), (8, 1))  # row-major: stride (8, 1)
    col = fx.make_layout((8, 8), (1, 8))  # column-major: stride (1, 8)
    for c in [(0, 1), (1, 0), (2, 3)]:
        crd = fx.make_coord(*c)
        print(f"coord {c}: row-major -> {fx.crd2idx(crd, row)}   column-major -> {fx.crd2idx(crd, col)}")


coord_to_index()

The coordinate `(2, 3)` is the same logical position both times — only the stride
changes where it lands: `2*8 + 3 = 19` row-major, `2*1 + 3*8 = 26` column-major.
`idx2crd` is the inverse direction (index back to coordinate).

## 3. Tiling with `logical_divide`

Real kernels never touch a whole tensor at once — they cut it into per-block and
per-thread tiles. `logical_divide` is the cut: it splits a layout by a *tiler*
layout, producing a hierarchical layout whose first mode is "inside one tile" and
whose second is "which tile." This is exactly the move `01-vectorAdd` makes to hand
each thread its slice of a 64-element block.

In [ ]:
@flyc.jit
def tiling():
    flat = fx.make_layout(64, 1)  # 64 elements, contiguous
    tiled = fx.logical_divide(flat, fx.make_layout(8, 1))
    print("flat        :", flat)
    print("divide by 8 :", tiled, "  (inner = 8 per tile, outer = 8 tiles)")
    print("inner mode  :", fx.select(tiled, [0]))  # one tile's layout
    print("rank        :", fx.rank(tiled))


tiling()

Read `(8, 8):(1, 8)` as **8 elements per tile** (first mode) × **8 tiles** (second
mode): `64 ÷ 8`. The stride `1` steps within a tile, `8` jumps to the next tile.
`fx.select(tiled, [0])` pulls out mode 0 — one tile's layout on its own.

## 4. The two kinds of tensor — memref vs coord

A layout describes *positions*. A **tensor** attaches that layout to something:

- A **memref tensor** is a layout over **memory** (a *memref* is MLIR's memory-backed
  tensor) — registers, LDS, or global. It has an element type, and reading it gives you
  *data*. This is what `make_rmem_tensor` and `rocdl.make_buffer_tensor` produce, and
  what the earlier notebooks copied.
- A **coord tensor** answers "*which logical coordinate am I holding?*". It is built on
  an **identity layout** — the layout analog of the identity function — where `crd2idx`
  returns the coordinate itself (an `IntTuple`) instead of collapsing it to a flat
  address. There is no memory behind it — only the coordinates — which is exactly why it
  has **no element type**. Coord tensors are how a kernel tracks *which* logical element
  a thread owns, the basis for predication and indexing.

You partition both the same way; one yields data, the other the coordinates that go
with it. The contrast below is the whole point: `crd2idx` collapses `(2, 3)` to the
index `19` through the ordinary layout, but returns `(2, 3)` through the identity one.

In [ ]:
@flyc.jit
def two_tensors():
    # memref tensor: a layout over register memory, carries f32 data
    mem = fx.make_rmem_tensor(fx.make_layout((4, 8), (8, 1)), fx.Float32)
    print("memref tensor:", mem, "| element_type:", mem.element_type)

    # coord tensor: a layout over coordinates, built on an identity layout
    crd = fx.make_view(fx.make_coord(0, 0), fx.make_identity_layout((4, 8)))
    print("coord  tensor:", crd)
    try:
        crd.element_type
    except TypeError as e:
        print("coord  tensor: element_type ->", e)

    # the difference is in the layout: ordinary collapses to an index, identity keeps the coordinate
    ordinary = fx.make_layout((4, 8), (8, 1))
    identity = fx.make_identity_layout((4, 8))
    c = fx.make_coord(2, 3)
    print("crd2idx (2,3): ordinary ->", fx.crd2idx(c, ordinary), "  identity ->", fx.crd2idx(c, identity))


two_tensors()

Reading the coord-tensor repr `Tensor<(0,0), (4,8):(1E0,1E1)>`: `(0,0)` is its base
coordinate, `(4,8)` the shape, and `(1E0, 1E1)` the *basis* stride — FlyDSL's notation
for the identity stride that keeps each coordinate symbolic instead of flattening it.
The memref's plain `(8, 1)` stride, by contrast, is a real step into memory.

That is the entire vocabulary the next notebook tiles and visualizes: shapes and
strides, coordinates and indices, `logical_divide` for tiling, and the two tensor
kinds you partition across threads.

---
**Next:** [`05_tiled_copy_and_swizzle`](05_tiled_copy_and_swizzle.ipynb) — thread-value
layouts, partitioning, the swizzle trick for LDS, and drawing layouts with `print_typst`.